# Momentum-Based Trading Framework

## Project Goal

This notebook builds a beginner-friendly but serious **momentum-based trading framework** in Python.

The central idea is simple:

> If an asset has recently moved strongly upward, it may continue moving upward for some period.  
> If it has recently moved downward, it may continue moving downward or be avoided.

This project walks through the full structure of a basic quantitative trading strategy:

1. Load price data  
2. Clean and inspect the data  
3. Create momentum indicators  
4. Generate trading signals  
5. Backtest the strategy  
6. Compare performance against buy-and-hold  
7. Evaluate returns, volatility, drawdown, and Sharpe ratio  
8. Add risk controls  
9. Discuss limitations and possible improvements  

This is an educational framework, not financial advice.

## 1. Import Libraries

We use standard Python data analysis libraries.

- `pandas` handles time-series data
- `numpy` handles numerical calculations
- `matplotlib` creates charts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

## 2. Load Price Data

This notebook can work in two ways:

### Option A: Use your own CSV file

If you have a CSV file with price data, place it in the same folder as this notebook and update the file name below.

The dataset should ideally contain:

- `date`
- `open`
- `high`
- `low`
- `close`
- `volume`

### Option B: Use simulated data

If no CSV file is available, the notebook creates a simple simulated price series so the full framework still runs.

In [ ]:
# Change this file name if you have your own data file.
# Example: "BTC-USD.csv", "AAPL.csv", or "crypto_prices.csv"
csv_file = "price_data.csv"

try:
    df = pd.read_csv(csv_file)
    print(f"Loaded data from {csv_file}")
except FileNotFoundError:
    print("No CSV file found. Creating simulated price data for demonstration.")

    np.random.seed(42)
    n = 1000
    dates = pd.date_range(start="2020-01-01", periods=n, freq="D")

    # Simulate returns with small drift and random noise
    returns = np.random.normal(loc=0.0005, scale=0.02, size=n)
    close = 100 * (1 + pd.Series(returns)).cumprod()

    df = pd.DataFrame({
        "date": dates,
        "open": close.shift(1).fillna(close.iloc[0]),
        "high": close * (1 + np.random.uniform(0.001, 0.02, size=n)),
        "low": close * (1 - np.random.uniform(0.001, 0.02, size=n)),
        "close": close,
        "volume": np.random.randint(1000, 10000, size=n)
    })

df.head()

## 3. Prepare and Inspect the Data

We standardize column names, convert the date column to datetime format, and sort the data chronologically.

This is important because trading strategies depend on correct time order.

In [ ]:
# Standardize column names
df.columns = [col.lower().strip().replace(" ", "_") for col in df.columns]

# Convert date column
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")
    df = df.set_index("date")
else:
    df = df.sort_index()

# Keep only rows with close prices
df = df.dropna(subset=["close"])

print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df["close"].plot(figsize=(12, 5))
plt.title("Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

## 4. Compute Daily Returns

The return measures the percentage change in price from one period to the next.

For daily data:

$$
r_t = \frac{P_t}{P_{t-1}} - 1
$$

where:

- $P_t$ is today's close price
- $P_{t-1}$ is yesterday's close price

In [ ]:
df["return"] = df["close"].pct_change()
df[["close", "return"]].head()

## 5. Create Momentum Indicators

Momentum can be measured in several ways.

Here we use three simple momentum features:

1. **Lookback return**  
   Percentage change over the past `lookback` periods.

2. **Short moving average**  
   Average price over a short window.

3. **Long moving average**  
   Average price over a longer window.

A common momentum idea is:

> Buy when short-term price behavior is stronger than long-term behavior.

In [ ]:
lookback = 20
short_window = 20
long_window = 100

df["momentum_return"] = df["close"].pct_change(lookback)
df["short_ma"] = df["close"].rolling(short_window).mean()
df["long_ma"] = df["close"].rolling(long_window).mean()

df[["close", "momentum_return", "short_ma", "long_ma"]].tail()

In [ ]:
df[["close", "short_ma", "long_ma"]].plot(figsize=(12, 5))
plt.title("Price with Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

## 6. Generate Trading Signals

We now create a simple momentum signal.

The strategy goes long when:

1. The lookback return is positive  
2. The short moving average is above the long moving average  

Otherwise, the strategy stays in cash.

Signal values:

- `1` = invested / long
- `0` = out of the market

In [ ]:
df["signal"] = np.where(
    (df["momentum_return"] > 0) & (df["short_ma"] > df["long_ma"]),
    1,
    0
)

df[["close", "momentum_return", "short_ma", "long_ma", "signal"]].tail()

In [ ]:
df["signal"].value_counts()

## 7. Avoid Look-Ahead Bias

This is one of the most important ideas in backtesting.

If today's signal is calculated using today's closing price, we cannot assume we traded before knowing that closing price.

So we shift the signal forward by one period:

> Today's position is based on yesterday's signal.

This prevents the strategy from using future information.

In [ ]:
df["position"] = df["signal"].shift(1)
df[["signal", "position"]].tail()

## 8. Backtest the Strategy

The strategy return is:

$$
\text{strategy return}_t = \text{position}_t \times \text{asset return}_t
$$

If the position is 1, we earn the asset return.  
If the position is 0, we earn 0 for that period.

In [ ]:
df["strategy_return"] = df["position"] * df["return"]

# Remove early rows with missing values from rolling windows
backtest = df.dropna().copy()

backtest[["return", "signal", "position", "strategy_return"]].head()

## 9. Compare Strategy vs Buy-and-Hold

We compare two approaches:

1. **Buy-and-hold**  
   Buy the asset at the beginning and hold it.

2. **Momentum strategy**  
   Hold the asset only when the momentum signal is active.

In [ ]:
backtest["buy_hold_equity"] = (1 + backtest["return"]).cumprod()
backtest["strategy_equity"] = (1 + backtest["strategy_return"]).cumprod()

backtest[["buy_hold_equity", "strategy_equity"]].plot(figsize=(12, 5))
plt.title("Momentum Strategy vs Buy-and-Hold")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## 10. Performance Metrics

We calculate several important trading metrics:

### Total Return

Overall return over the backtest.

### Annualized Return

Estimated yearly return.

### Annualized Volatility

Estimated yearly risk.

### Sharpe Ratio

Return per unit of risk.

### Maximum Drawdown

Largest peak-to-trough loss.

These metrics help us compare performance and risk.

In [ ]:
def max_drawdown(equity_curve):
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1
    return drawdown.min()

def performance_metrics(returns, periods_per_year=252):
    returns = returns.dropna()
    equity = (1 + returns).cumprod()

    total_return = equity.iloc[-1] - 1
    annual_return = equity.iloc[-1] ** (periods_per_year / len(returns)) - 1
    annual_volatility = returns.std() * np.sqrt(periods_per_year)

    if annual_volatility != 0:
        sharpe = annual_return / annual_volatility
    else:
        sharpe = np.nan

    mdd = max_drawdown(equity)

    return {
        "Total Return": total_return,
        "Annualized Return": annual_return,
        "Annualized Volatility": annual_volatility,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": mdd
    }

strategy_metrics = performance_metrics(backtest["strategy_return"])
buy_hold_metrics = performance_metrics(backtest["return"])

metrics_df = pd.DataFrame({
    "Momentum Strategy": strategy_metrics,
    "Buy and Hold": buy_hold_metrics
})

metrics_df

In [ ]:
(metrics_df * 100).round(2)

Note: The Sharpe ratio should not be multiplied by 100. The previous table multiplies all values for compact display, so read the Sharpe row separately from the percentage metrics if needed.

## 11. Drawdown Analysis

Drawdown shows how much the portfolio falls from its previous high.

This is one of the most important risk measures in trading because a strategy can have high returns but still suffer painful losses.

In [ ]:
backtest["strategy_running_max"] = backtest["strategy_equity"].cummax()
backtest["strategy_drawdown"] = backtest["strategy_equity"] / backtest["strategy_running_max"] - 1

backtest["buy_hold_running_max"] = backtest["buy_hold_equity"].cummax()
backtest["buy_hold_drawdown"] = backtest["buy_hold_equity"] / backtest["buy_hold_running_max"] - 1

backtest[["strategy_drawdown", "buy_hold_drawdown"]].plot(figsize=(12, 5))
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.show()

## 12. Add Transaction Costs

Real trading is not free.

Every trade may involve:

- Exchange fees
- Bid-ask spread
- Slippage
- Market impact

Here we add a simple transaction cost whenever the position changes.

In [ ]:
transaction_cost = 0.001  # 0.10% per trade

backtest["trade"] = backtest["position"].diff().abs()
backtest["cost"] = backtest["trade"] * transaction_cost

backtest["strategy_return_after_cost"] = backtest["strategy_return"] - backtest["cost"]
backtest["strategy_equity_after_cost"] = (1 + backtest["strategy_return_after_cost"]).cumprod()

backtest[["strategy_equity", "strategy_equity_after_cost", "buy_hold_equity"]].plot(figsize=(12, 5))
plt.title("Strategy Performance Before and After Transaction Costs")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

In [ ]:
after_cost_metrics = performance_metrics(backtest["strategy_return_after_cost"])

pd.DataFrame({
    "Momentum Before Cost": strategy_metrics,
    "Momentum After Cost": after_cost_metrics,
    "Buy and Hold": buy_hold_metrics
})

## 13. Add a Simple Stop-Loss Rule

A stop-loss exits the position if the asset falls too much from the entry price.

This is a simplified example. Real stop-loss logic can be more complex, especially with intraday data.

In [ ]:
stop_loss = 0.08  # 8%

bt = backtest.copy()
bt["position_stop"] = 0
bt["entry_price"] = np.nan

in_trade = False
entry_price = None

for i in range(len(bt)):
    current_signal = bt["signal"].iloc[i]
    current_price = bt["close"].iloc[i]

    if not in_trade and current_signal == 1:
        in_trade = True
        entry_price = current_price
        bt.iloc[i, bt.columns.get_loc("position_stop")] = 1
        bt.iloc[i, bt.columns.get_loc("entry_price")] = entry_price

    elif in_trade:
        loss_from_entry = current_price / entry_price - 1

        if loss_from_entry <= -stop_loss:
            in_trade = False
            entry_price = None
            bt.iloc[i, bt.columns.get_loc("position_stop")] = 0
        elif current_signal == 0:
            in_trade = False
            entry_price = None
            bt.iloc[i, bt.columns.get_loc("position_stop")] = 0
        else:
            bt.iloc[i, bt.columns.get_loc("position_stop")] = 1
            bt.iloc[i, bt.columns.get_loc("entry_price")] = entry_price

# Shift position to avoid look-ahead bias
bt["position_stop"] = bt["position_stop"].shift(1)
bt["strategy_return_stop"] = bt["position_stop"] * bt["return"]
bt["strategy_equity_stop"] = (1 + bt["strategy_return_stop"].fillna(0)).cumprod()

bt[["strategy_equity", "strategy_equity_stop", "buy_hold_equity"]].plot(figsize=(12, 5))
plt.title("Momentum Strategy With and Without Stop-Loss")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

In [ ]:
stop_metrics = performance_metrics(bt["strategy_return_stop"])

pd.DataFrame({
    "Momentum Strategy": strategy_metrics,
    "Momentum With Stop-Loss": stop_metrics,
    "Buy and Hold": buy_hold_metrics
})

## 14. Parameter Testing

A trading strategy should not depend on one arbitrary parameter choice.

We can test different lookback windows to see whether the strategy is stable or only works for one lucky setting.

This is a simple parameter scan.

In [ ]:
def run_momentum_backtest(data, lookback=20, short_window=20, long_window=100):
    temp = data.copy()

    temp["return"] = temp["close"].pct_change()
    temp["momentum_return"] = temp["close"].pct_change(lookback)
    temp["short_ma"] = temp["close"].rolling(short_window).mean()
    temp["long_ma"] = temp["close"].rolling(long_window).mean()

    temp["signal"] = np.where(
        (temp["momentum_return"] > 0) & (temp["short_ma"] > temp["long_ma"]),
        1,
        0
    )

    temp["position"] = temp["signal"].shift(1)
    temp["strategy_return"] = temp["position"] * temp["return"]
    temp = temp.dropna()

    if len(temp) == 0:
        return None

    metrics = performance_metrics(temp["strategy_return"])
    metrics["Lookback"] = lookback
    metrics["Short MA"] = short_window
    metrics["Long MA"] = long_window

    return metrics

results = []

for lb in [5, 10, 20, 40, 60]:
    result = run_momentum_backtest(df, lookback=lb, short_window=lb, long_window=100)
    if result is not None:
        results.append(result)

param_results = pd.DataFrame(results)
param_results = param_results[["Lookback", "Short MA", "Long MA", "Total Return", "Annualized Return", "Annualized Volatility", "Sharpe Ratio", "Max Drawdown"]]
param_results

In [ ]:
param_results.set_index("Lookback")["Sharpe Ratio"].plot(kind="bar")
plt.title("Sharpe Ratio by Lookback Window")
plt.xlabel("Lookback Window")
plt.ylabel("Sharpe Ratio")
plt.xticks(rotation=0)
plt.show()

## 15. Interpreting the Framework

This notebook created a complete but simple momentum trading framework.

The key logic was:

1. Measure recent price strength  
2. Confirm trend using moving averages  
3. Enter the market only when momentum is positive  
4. Avoid look-ahead bias by shifting positions  
5. Compare results with buy-and-hold  
6. Evaluate risk using volatility and drawdown  
7. Include transaction costs  
8. Test parameter sensitivity  

A strong framework is not just about high return. It must also be tested for robustness, risk, and realistic trading conditions.

## 16. Limitations

This strategy is intentionally simple. It has important limitations:

- It uses only price-based momentum.
- It does not include volume-based confirmation.
- It does not include market regime detection.
- It does not optimize position sizing.
- It does not use walk-forward validation.
- Transaction costs are simplified.
- Stop-loss execution is simplified.
- The simulated data example is not a substitute for real historical data.

In real strategy development, the most important question is not whether a strategy worked once, but whether it remains stable across assets, time periods, and market regimes.

## 17. Possible Improvements

This framework can be extended in many directions:

### Better signals

- RSI momentum
- MACD confirmation
- Volatility-adjusted momentum
- Breakout signals
- Cross-sectional momentum across multiple assets

### Better risk management

- Volatility targeting
- Dynamic position sizing
- Trailing stop-loss
- Maximum exposure limits
- Drawdown-based de-risking

### Better validation

- Train-test split by time
- Walk-forward testing
- Out-of-sample testing
- Parameter robustness heatmaps
- Monte Carlo resampling

### Machine learning extension

A machine learning model could use momentum features as inputs and learn when momentum is more likely to continue or fail.

## 18. Final Conclusion

Momentum is one of the most important ideas in systematic trading.

This notebook showed how to turn that idea into a structured Python framework:

- Define the trading idea clearly
- Convert the idea into signals
- Backtest the signals carefully
- Avoid look-ahead bias
- Compare with a benchmark
- Measure risk and return
- Add realism through transaction costs
- Test parameter stability

The result is not a finished production strategy, but it is a strong foundation for developing, testing, and improving systematic trading strategies.